# Experiment: Model Compute Report

Calculate the values usually reported in a paper for this DuoDiT/DiT model: FLOPs, parameters, GPU, and training time.
Run the notebook top to bottom after editing the configuration cell.


## What This Reports

- Parameters: total, trainable, frozen, and component-level counts.
- FLOPs: measured forward-pass FLOPs for a dummy 256x256-image latent input.
- GPU usage per epoch: live CUDA utilization, memory utilization, VRAM used, PyTorch allocated/reserved memory, and power.
- Time per epoch: live measured seconds per step scaled to your configured steps per epoch.


In [ ]:
from __future__ import annotations

import contextlib
import io
import math
import statistics
import subprocess
import sys
import threading
import time
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    display = print

import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "models.py").exists():
    REPO_ROOT = Path("/Users/mrmc/Documents/GitHub/DuoDiT")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")


In [ ]:
# Edit this cell for the run you want to report.
CONFIG = {
    "model_name": "DiT-XL/2",
    "image_size": 256,              # Paper image resolution: 256x256.
    "num_classes": 1000,
    "training_mode": "x2_finetune", # "x2_finetune" or "full"
    "checkpoint_path": None,        # Optional: "/path/to/checkpoint.pt"
    "checkpoint_state_key": "ema",  # Usually "ema" or "model"
    "force_local_timm_vit": True,    # Avoid network downloads; architecture is unchanged.

    # FLOPs benchmark.
    "run_flop_profile": True,
    "flop_batch_size": 1,
    "flop_device": "auto",          # "auto", "cuda", "mps", or "cpu"

    # Live epoch benchmark. Set benchmark_batch_size to your real per-GPU batch size.
    "benchmark_training": True,
    "benchmark_batch_size": 1,
    "benchmark_warmup_steps": 5,
    "benchmark_epochs": 1,
    "benchmark_steps_per_epoch": 200, # Set to None to benchmark the full configured epoch.
    "gpu_sample_interval_sec": 0.05,
    "benchmark_randomize_zero_output_layer": True, # Helps timing when no trained checkpoint is loaded.

    # Epoch definition. Prefer steps_per_epoch if you know it.
    "global_batch_size": 256,
    "steps_per_epoch": 200,
    "dataset_size": None,            # Used only if steps_per_epoch is None.
    "epochs": 400,
    "planned_train_steps": None,      # Optional override for total training-time estimate.
    "training_flop_multiplier": 3.0,  # Approx. forward + backward + optimizer.
}

CONFIG


In [ ]:
def human_int(value: int | float | None) -> str:
    if value is None:
        return "n/a"
    return f"{int(value):,}"


def human_float(value: float | None, digits: int = 2) -> str:
    if value is None or math.isnan(value):
        return "n/a"
    return f"{value:,.{digits}f}"


def format_bytes(num_bytes: int | float | None) -> str:
    if num_bytes is None:
        return "n/a"
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024
    return f"{value:.2f} TB"


def format_duration(seconds: float | None) -> str:
    if seconds is None or math.isnan(seconds) or seconds < 0:
        return "n/a"
    seconds = int(round(seconds))
    days, rem = divmod(seconds, 86_400)
    hours, rem = divmod(rem, 3_600)
    minutes, secs = divmod(rem, 60)
    parts = []
    if days:
        parts.append(f"{days}d")
    if hours or days:
        parts.append(f"{hours}h")
    if minutes or hours or days:
        parts.append(f"{minutes}m")
    parts.append(f"{secs}s")
    return " ".join(parts)


def as_table(records):
    try:
        import pandas as pd
        return pd.DataFrame(records)
    except Exception:
        return records


def choose_device(requested: str = "auto") -> torch.device:
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CONFIG['flop_device']='cuda' but CUDA is not available.")
    if requested == "mps" and not (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()):
        raise RuntimeError("CONFIG['flop_device']='mps' but MPS is not available.")
    return torch.device(requested)


In [ ]:
def gpu_report() -> list[dict]:
    rows = []
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            rows.append({
                "backend": "cuda",
                "index": index,
                "name": props.name,
                "memory_gb": round(props.total_memory / 1024**3, 2),
                "compute_capability": f"{props.major}.{props.minor}",
                "multi_processor_count": props.multi_processor_count,
            })
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        rows.append({
            "backend": "mps",
            "index": 0,
            "name": "Apple Metal Performance Shaders",
            "memory_gb": None,
            "compute_capability": None,
            "multi_processor_count": None,
        })
    else:
        rows.append({
            "backend": "cpu",
            "index": 0,
            "name": "No GPU detected",
            "memory_gb": None,
            "compute_capability": None,
            "multi_processor_count": None,
        })

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader",
            ],
            check=False,
            capture_output=True,
            text=True,
            timeout=5,
        )
        if result.returncode == 0 and result.stdout.strip():
            for row, line in zip(rows, result.stdout.strip().splitlines()):
                name, memory_total, driver = [part.strip() for part in line.split(",", 2)]
                row["nvidia_smi_name"] = name
                row["nvidia_smi_memory"] = memory_total
                row["driver_version"] = driver
    except Exception:
        pass
    return rows


gpu_rows = gpu_report()
display(as_table(gpu_rows))


In [ ]:
from models import DiT_models


def build_model(config: dict) -> torch.nn.Module:
    image_size = int(config["image_size"])
    assert image_size % 8 == 0, "image_size must be divisible by 8 for the VAE latent input."
    latent_size = image_size // 8
    model_name = config["model_name"]

    if config.get("force_local_timm_vit", True):
        import timm
        original_create_model = timm.create_model

        def create_model_no_download(*args, **kwargs):
            kwargs["pretrained"] = False
            return original_create_model(*args, **kwargs)

        timm.create_model = create_model_no_download
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                model = DiT_models[model_name](
                    input_size=latent_size,
                    num_classes=int(config["num_classes"]),
                )
        finally:
            timm.create_model = original_create_model
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            model = DiT_models[model_name](
                input_size=latent_size,
                num_classes=int(config["num_classes"]),
            )

    ckpt_path = config.get("checkpoint_path")
    if ckpt_path:
        checkpoint = torch.load(ckpt_path, map_location="cpu")
        state_key = config.get("checkpoint_state_key")
        if isinstance(checkpoint, dict) and state_key in checkpoint:
            state = checkpoint[state_key]
        elif isinstance(checkpoint, dict):
            state = checkpoint.get("ema") or checkpoint.get("model") or checkpoint.get("state_dict") or checkpoint
        else:
            state = checkpoint
        state = {key.removeprefix("module."): value for key, value in state.items()}
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"Loaded checkpoint: {ckpt_path}")
        print(f"Missing keys: {len(missing):,}; unexpected keys: {len(unexpected):,}")

    return model


def apply_training_mode(model: torch.nn.Module, mode: str) -> None:
    mode = mode.lower()
    if mode == "full":
        for param in model.parameters():
            param.requires_grad = True
        return

    if mode != "x2_finetune":
        raise ValueError("training_mode must be 'x2_finetune' or 'full'.")

    for param in model.parameters():
        param.requires_grad = False

    for module_name in ("x2_embedder", "x2_vit_block", "x2_vit_proj_in", "x2_vit_proj_out", "final_layer"):
        module = getattr(model, module_name, None)
        if module is not None:
            for param in module.parameters():
                param.requires_grad = True

    if hasattr(model, "x2_cls_tokens"):
        model.x2_cls_tokens.requires_grad = True


model = build_model(CONFIG)
apply_training_mode(model, CONFIG["training_mode"])
model.eval()

print(model.__class__.__name__)
print(f"latent input size: {CONFIG['image_size'] // 8} x {CONFIG['image_size'] // 8}")


In [ ]:
def parameter_rows(model: torch.nn.Module) -> list[dict]:
    rows = []
    grouped: dict[str, dict[str, int]] = {}
    for name, param in model.named_parameters():
        top = name.split(".", 1)[0]
        grouped.setdefault(top, {"total": 0, "trainable": 0})
        grouped[top]["total"] += param.numel()
        if param.requires_grad:
            grouped[top]["trainable"] += param.numel()

    total = sum(item["total"] for item in grouped.values())
    trainable = sum(item["trainable"] for item in grouped.values())
    rows.append({
        "component": "TOTAL",
        "total_params": total,
        "trainable_params": trainable,
        "frozen_params": total - trainable,
        "trainable_percent": 100 * trainable / total if total else 0,
    })
    for component, counts in sorted(grouped.items()):
        total_component = counts["total"]
        trainable_component = counts["trainable"]
        rows.append({
            "component": component,
            "total_params": total_component,
            "trainable_params": trainable_component,
            "frozen_params": total_component - trainable_component,
            "trainable_percent": 100 * trainable_component / total_component if total_component else 0,
        })
    return rows


param_rows = parameter_rows(model)
display(as_table(param_rows))

total_params = param_rows[0]["total_params"]
trainable_params = param_rows[0]["trainable_params"]
print(f"Paper parameter count: {total_params / 1e6:.2f}M total")
print(f"Trainable parameter count: {trainable_params / 1e6:.2f}M")


In [ ]:
def dummy_inputs(config: dict, device: torch.device, batch_size: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    latent_size = int(config["image_size"]) // 8
    num_classes = int(config["num_classes"])
    x = torch.randn(batch_size, 4, latent_size, latent_size, device=device)
    t = torch.randint(0, 1000, (batch_size,), device=device)
    y = torch.randint(0, num_classes, (batch_size,), device=device)
    return x, t, y


def profile_with_fvcore(model: torch.nn.Module, config: dict, device: torch.device, batch_size: int) -> dict | None:
    try:
        from fvcore.nn import FlopCountAnalysis
    except Exception:
        return None

    work_model = model.to(device).eval()
    inputs = dummy_inputs(config, device, batch_size)
    with torch.inference_mode(), contextlib.redirect_stdout(io.StringIO()):
        analysis = FlopCountAnalysis(work_model, inputs)
        total_flops = int(analysis.total())

    return {
        "method": "fvcore.nn.FlopCountAnalysis",
        "flops": total_flops,
        "unsupported_ops": dict(analysis.unsupported_ops()),
    }


def profile_with_torch_profiler(model: torch.nn.Module, config: dict, device: torch.device, batch_size: int) -> dict:
    from torch.profiler import ProfilerActivity, profile

    work_model = model.to(device).eval()
    inputs = dummy_inputs(config, device, batch_size)
    activities = [ProfilerActivity.CPU]
    if device.type == "cuda":
        activities.append(ProfilerActivity.CUDA)

    with torch.inference_mode(), contextlib.redirect_stdout(io.StringIO()):
        _ = work_model(*inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()

    with torch.inference_mode(), contextlib.redirect_stdout(io.StringIO()):
        with profile(activities=activities, with_flops=True, record_shapes=False) as prof:
            _ = work_model(*inputs)
            if device.type == "cuda":
                torch.cuda.synchronize()

    total_flops = 0
    for event in prof.key_averages():
        if event.flops is not None:
            total_flops += event.flops

    return {
        "method": "torch.profiler.profile(with_flops=True)",
        "flops": int(total_flops),
        "unsupported_ops": None,
    }


def measure_forward_flops(model: torch.nn.Module, config: dict) -> dict:
    batch_size = int(config["flop_batch_size"])
    device = choose_device(config.get("flop_device", "auto"))
    fvcore_result = profile_with_fvcore(model, config, device, batch_size)
    result = fvcore_result or profile_with_torch_profiler(model, config, device, batch_size)
    result.update({
        "device": str(device),
        "batch_size": batch_size,
        "flops_per_sample": result["flops"] / batch_size,
        "gflops_per_sample": result["flops"] / batch_size / 1e9,
    })
    return result


In [ ]:
if CONFIG["run_flop_profile"]:
    flop_result = measure_forward_flops(model, CONFIG)
else:
    flop_result = {
        "method": "skipped",
        "flops": None,
        "flops_per_sample": None,
        "gflops_per_sample": None,
        "device": None,
        "batch_size": CONFIG["flop_batch_size"],
        "unsupported_ops": None,
    }

display(as_table([flop_result]))
print(f"Paper forward FLOPs: {human_float(flop_result['gflops_per_sample'])} GFLOPs per sample")


In [ ]:
def parse_float(value: str) -> float | None:
    value = value.strip()
    if value in {"", "N/A", "[N/A]"}:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def query_nvidia_smi() -> list[dict]:
    query = "index,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw"
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                f"--query-gpu={query}",
                "--format=csv,noheader,nounits",
            ],
            check=False,
            capture_output=True,
            text=True,
            timeout=5,
        )
    except Exception:
        return []

    if result.returncode != 0 or not result.stdout.strip():
        return []

    rows = []
    for line in result.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 7:
            continue
        rows.append({
            "timestamp": time.time(),
            "index": int(parts[0]),
            "name": parts[1],
            "gpu_util_percent": parse_float(parts[2]),
            "memory_util_percent": parse_float(parts[3]),
            "memory_used_mb": parse_float(parts[4]),
            "memory_total_mb": parse_float(parts[5]),
            "power_w": parse_float(parts[6]),
        })
    return rows


class GpuUsageSampler:
    def __init__(self, interval_sec: float = 0.05):
        self.interval_sec = interval_sec
        self.samples: list[dict] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def start(self):
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self) -> list[dict]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)
        return self.samples

    def _run(self):
        while not self._stop.is_set():
            self.samples.extend(query_nvidia_smi())
            self._stop.wait(self.interval_sec)


def summarize_gpu_samples(samples: list[dict], device: torch.device) -> dict:
    if not samples:
        return {
            "gpu_usage_samples": 0,
            "mean_gpu_util_percent": None,
            "max_gpu_util_percent": None,
            "mean_memory_util_percent": None,
            "max_memory_util_percent": None,
            "max_nvidia_smi_memory_used_gb": None,
            "mean_power_w": None,
            "max_power_w": None,
        }

    device_index = device.index if device.index is not None else torch.cuda.current_device() if device.type == "cuda" else None
    filtered = [sample for sample in samples if sample["index"] == device_index] if device_index is not None else samples
    if not filtered:
        filtered = samples

    def values(key: str) -> list[float]:
        return [sample[key] for sample in filtered if sample.get(key) is not None]

    gpu_util = values("gpu_util_percent")
    memory_util = values("memory_util_percent")
    memory_used = values("memory_used_mb")
    power = values("power_w")
    return {
        "gpu_usage_samples": len(filtered),
        "mean_gpu_util_percent": statistics.mean(gpu_util) if gpu_util else None,
        "max_gpu_util_percent": max(gpu_util) if gpu_util else None,
        "mean_memory_util_percent": statistics.mean(memory_util) if memory_util else None,
        "max_memory_util_percent": max(memory_util) if memory_util else None,
        "max_nvidia_smi_memory_used_gb": max(memory_used) / 1024 if memory_used else None,
        "mean_power_w": statistics.mean(power) if power else None,
        "max_power_w": max(power) if power else None,
    }


def torch_memory_stats(device: torch.device) -> dict:
    if device.type == "cuda":
        return {
            "max_torch_memory_allocated_gb": torch.cuda.max_memory_allocated(device) / 1024**3,
            "max_torch_memory_reserved_gb": torch.cuda.max_memory_reserved(device) / 1024**3,
        }
    if device.type == "mps":
        return {
            "max_torch_memory_allocated_gb": torch.mps.current_allocated_memory() / 1024**3,
            "max_torch_memory_reserved_gb": None,
        }
    return {
        "max_torch_memory_allocated_gb": None,
        "max_torch_memory_reserved_gb": None,
    }


def steps_per_epoch_from_config(config: dict) -> int | None:
    if config.get("steps_per_epoch") is not None:
        return int(config["steps_per_epoch"])
    dataset_size = config.get("dataset_size")
    global_batch_size = config.get("global_batch_size")
    if dataset_size is not None and global_batch_size:
        return math.ceil(int(dataset_size) / int(global_batch_size))
    return None


def planned_steps_from_config(config: dict) -> int | None:
    if config.get("planned_train_steps") is not None:
        return int(config["planned_train_steps"])
    steps_per_epoch = steps_per_epoch_from_config(config)
    epochs = config.get("epochs")
    if steps_per_epoch is not None and epochs is not None:
        return int(steps_per_epoch) * int(epochs)
    return None


def training_step(model: torch.nn.Module, optimizer: torch.optim.Optimizer, config: dict, device: torch.device, batch_size: int) -> float:
    x, t, y = dummy_inputs(config, device, batch_size)
    optimizer.zero_grad(set_to_none=True)
    with contextlib.redirect_stdout(io.StringIO()):
        output = model(x, t, y)
    target = torch.randn_like(output)
    loss = torch.nn.functional.mse_loss(output.float(), target.float())
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu())


def make_zero_output_layer_trainable_for_benchmark(model: torch.nn.Module, config: dict) -> None:
    if not config.get("benchmark_randomize_zero_output_layer", True):
        return
    final_layer = getattr(model, "final_layer", None)
    linear = getattr(final_layer, "linear", None)
    if linear is None:
        return
    with torch.no_grad():
        if torch.count_nonzero(linear.weight).item() == 0:
            torch.nn.init.normal_(linear.weight, mean=0.0, std=0.02)
        if linear.bias is not None and torch.count_nonzero(linear.bias).item() == 0:
            linear.bias.normal_(mean=0.0, std=0.02)


def benchmark_training_epochs(model: torch.nn.Module, config: dict) -> tuple[list[dict], dict]:
    device = choose_device(config.get("flop_device", "auto"))
    batch_size = int(config["benchmark_batch_size"])
    warmup_steps = int(config["benchmark_warmup_steps"])
    benchmark_epochs = int(config["benchmark_epochs"])
    configured_steps_per_epoch = steps_per_epoch_from_config(config)
    benchmark_steps_per_epoch = config.get("benchmark_steps_per_epoch")
    if benchmark_steps_per_epoch is None:
        if configured_steps_per_epoch is None:
            raise ValueError("Set CONFIG['steps_per_epoch'] or CONFIG['benchmark_steps_per_epoch'].")
        benchmark_steps_per_epoch = configured_steps_per_epoch
    benchmark_steps_per_epoch = int(benchmark_steps_per_epoch)

    work_model = model.to(device).train()
    make_zero_output_layer_trainable_for_benchmark(work_model, config)
    trainable = [param for param in work_model.parameters() if param.requires_grad]
    if not trainable:
        raise RuntimeError("No trainable parameters found. Check CONFIG['training_mode'].")

    optimizer = torch.optim.AdamW(trainable, lr=1e-4, weight_decay=0)

    for _ in range(warmup_steps):
        training_step(work_model, optimizer, config, device, batch_size)
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    epoch_rows = []
    for epoch_index in range(1, benchmark_epochs + 1):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)

        sampler = GpuUsageSampler(float(config["gpu_sample_interval_sec"]))
        sampler.start()
        losses = []

        if device.type == "cuda":
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            start_event.record()
            for _ in range(benchmark_steps_per_epoch):
                losses.append(training_step(work_model, optimizer, config, device, batch_size))
            end_event.record()
            torch.cuda.synchronize(device)
            elapsed_seconds = start_event.elapsed_time(end_event) / 1000
        else:
            start = time.perf_counter()
            for _ in range(benchmark_steps_per_epoch):
                losses.append(training_step(work_model, optimizer, config, device, batch_size))
            if device.type == "mps":
                torch.mps.synchronize()
            elapsed_seconds = time.perf_counter() - start

        samples = sampler.stop()
        seconds_per_step = elapsed_seconds / benchmark_steps_per_epoch
        estimated_full_epoch_seconds = (
            seconds_per_step * configured_steps_per_epoch
            if configured_steps_per_epoch is not None
            else elapsed_seconds
        )
        row = {
            "epoch": epoch_index,
            "image_size": f"{config['image_size']}x{config['image_size']}",
            "device": str(device),
            "batch_size_per_gpu": batch_size,
            "measured_steps": benchmark_steps_per_epoch,
            "configured_steps_per_epoch": configured_steps_per_epoch,
            "elapsed_seconds_measured": elapsed_seconds,
            "seconds_per_step": seconds_per_step,
            "steps_per_sec": 1 / seconds_per_step if seconds_per_step else None,
            "estimated_seconds_per_epoch": estimated_full_epoch_seconds,
            "estimated_time_per_epoch": format_duration(estimated_full_epoch_seconds),
            "last_loss": losses[-1] if losses else None,
        }
        row.update(torch_memory_stats(device))
        row.update(summarize_gpu_samples(samples, device))
        epoch_rows.append(row)

    def mean_present(key: str) -> float | None:
        vals = [row[key] for row in epoch_rows if row.get(key) is not None]
        return statistics.mean(vals) if vals else None

    def max_present(key: str) -> float | None:
        vals = [row[key] for row in epoch_rows if row.get(key) is not None]
        return max(vals) if vals else None

    summary = {
        "method": "live epoch-shaped forward+backward+AdamW benchmark",
        "device": str(device),
        "image_size": f"{config['image_size']}x{config['image_size']}",
        "batch_size_per_gpu": batch_size,
        "configured_steps_per_epoch": configured_steps_per_epoch,
        "benchmark_epochs": benchmark_epochs,
        "benchmark_steps_per_epoch": benchmark_steps_per_epoch,
        "mean_seconds_per_step": mean_present("seconds_per_step"),
        "mean_steps_per_sec": mean_present("steps_per_sec"),
        "mean_estimated_seconds_per_epoch": mean_present("estimated_seconds_per_epoch"),
        "estimated_time_per_epoch": format_duration(mean_present("estimated_seconds_per_epoch")),
        "mean_gpu_util_percent": mean_present("mean_gpu_util_percent"),
        "max_gpu_util_percent": max_present("max_gpu_util_percent"),
        "mean_memory_util_percent": mean_present("mean_memory_util_percent"),
        "max_memory_util_percent": max_present("max_memory_util_percent"),
        "max_torch_memory_allocated_gb": max_present("max_torch_memory_allocated_gb"),
        "max_torch_memory_reserved_gb": max_present("max_torch_memory_reserved_gb"),
        "max_nvidia_smi_memory_used_gb": max_present("max_nvidia_smi_memory_used_gb"),
        "mean_power_w": mean_present("mean_power_w"),
        "max_power_w": max_present("max_power_w"),
    }
    return epoch_rows, summary


if CONFIG["benchmark_training"]:
    epoch_benchmark_rows, train_benchmark_result = benchmark_training_epochs(model, CONFIG)
else:
    epoch_benchmark_rows, train_benchmark_result = [], {
        "method": "skipped",
        "mean_seconds_per_step": None,
        "mean_estimated_seconds_per_epoch": None,
        "mean_gpu_util_percent": None,
        "max_gpu_util_percent": None,
    }

display(as_table(epoch_benchmark_rows))
display(as_table([train_benchmark_result]))
print(f"Live time per epoch: {train_benchmark_result.get('estimated_time_per_epoch', 'n/a')}")
print(f"Live GPU utilization per epoch: mean {human_float(train_benchmark_result.get('mean_gpu_util_percent'))}%, max {human_float(train_benchmark_result.get('max_gpu_util_percent'))}%")
print(f"Live VRAM per epoch: peak torch allocated {human_float(train_benchmark_result.get('max_torch_memory_allocated_gb'))} GB; peak nvidia-smi used {human_float(train_benchmark_result.get('max_nvidia_smi_memory_used_gb'))} GB")


In [ ]:
effective_train_steps = planned_steps_from_config(CONFIG)
configured_steps_per_epoch = steps_per_epoch_from_config(CONFIG)
seconds_per_step = train_benchmark_result.get("mean_seconds_per_step")
seconds_per_epoch = train_benchmark_result.get("mean_estimated_seconds_per_epoch")

estimated_training_seconds = effective_train_steps * seconds_per_step if effective_train_steps and seconds_per_step else None

forward_flops_per_sample = flop_result.get("flops_per_sample")
estimated_training_flops = None
if forward_flops_per_sample and effective_train_steps:
    estimated_training_flops = (
        forward_flops_per_sample
        * int(CONFIG["global_batch_size"])
        * int(effective_train_steps)
        * float(CONFIG["training_flop_multiplier"])
    )

gpu_name = gpu_rows[0]["name"] if gpu_rows else "n/a"
summary = {
    "model": CONFIG["model_name"],
    "image_size": f"{CONFIG['image_size']}x{CONFIG['image_size']}",
    "parameters_total": total_params,
    "parameters_trainable": trainable_params,
    "forward_gflops_per_sample": flop_result.get("gflops_per_sample"),
    "flop_method": flop_result.get("method"),
    "gpu": gpu_name,
    "steps_per_epoch": configured_steps_per_epoch,
    "time_per_epoch": format_duration(seconds_per_epoch),
    "live_seconds_per_step": seconds_per_step,
    "live_steps_per_sec": train_benchmark_result.get("mean_steps_per_sec"),
    "planned_train_steps": effective_train_steps,
    "estimated_total_training_time": format_duration(estimated_training_seconds),
    "mean_gpu_util_percent_per_epoch": train_benchmark_result.get("mean_gpu_util_percent"),
    "max_gpu_util_percent_per_epoch": train_benchmark_result.get("max_gpu_util_percent"),
    "mean_vram_util_percent_per_epoch": train_benchmark_result.get("mean_memory_util_percent"),
    "max_vram_util_percent_per_epoch": train_benchmark_result.get("max_memory_util_percent"),
    "peak_torch_vram_allocated_gb_per_epoch": train_benchmark_result.get("max_torch_memory_allocated_gb"),
    "peak_torch_vram_reserved_gb_per_epoch": train_benchmark_result.get("max_torch_memory_reserved_gb"),
    "peak_nvidia_smi_vram_used_gb_per_epoch": train_benchmark_result.get("max_nvidia_smi_memory_used_gb"),
    "mean_power_w_per_epoch": train_benchmark_result.get("mean_power_w"),
    "max_power_w_per_epoch": train_benchmark_result.get("max_power_w"),
    "training_flops_estimate": estimated_training_flops,
    "training_pflops_estimate": estimated_training_flops / 1e15 if estimated_training_flops else None,
}

display(as_table([summary]))


In [ ]:
paper_ready = {
    "Image size": summary["image_size"],
    "FLOPs": (
        f"{summary['forward_gflops_per_sample']:.2f} GFLOPs per forward pass"
        if summary["forward_gflops_per_sample"] is not None else "n/a"
    ),
    "Parameters": (
        f"{summary['parameters_total'] / 1e6:.2f}M total; "
        f"{summary['parameters_trainable'] / 1e6:.2f}M trainable"
    ),
    "GPU": summary["gpu"],
    "Time per epoch": (
        f"{summary['time_per_epoch']} "
        f"({summary['live_seconds_per_step']:.4f} sec/step x {summary['steps_per_epoch']:,} steps/epoch)"
        if summary["live_seconds_per_step"] is not None and summary["steps_per_epoch"] is not None
        else "n/a; set CONFIG['steps_per_epoch'] or dataset_size+global_batch_size"
    ),
    "GPU usage per epoch": (
        f"mean {summary['mean_gpu_util_percent_per_epoch']:.1f}% / max {summary['max_gpu_util_percent_per_epoch']:.1f}% GPU; "
        f"mean {summary['mean_vram_util_percent_per_epoch']:.1f}% / max {summary['max_vram_util_percent_per_epoch']:.1f}% VRAM"
        if summary["mean_gpu_util_percent_per_epoch"] is not None
        else "n/a"
    ),
    "VRAM per epoch": (
        f"peak {summary['peak_torch_vram_allocated_gb_per_epoch']:.2f} GB allocated; "
        f"{summary['peak_torch_vram_reserved_gb_per_epoch']:.2f} GB reserved"
        if summary["peak_torch_vram_allocated_gb_per_epoch"] is not None
        else "n/a"
    ),
}

print("Paper-ready values")
for key, value in paper_ready.items():
    print(f"{key}: {value}")

if summary["estimated_total_training_time"] != "n/a":
    print(f"Estimated total training time: {summary['estimated_total_training_time']}")
if summary["training_pflops_estimate"] is not None:
    print(f"Estimated training compute: {summary['training_pflops_estimate']:.2f} PFLOPs")

paper_ready


## Reporting Notes

- Image size is fixed to `256`, so the profiled latent tensor is `4 x 32 x 32`.
- Report forward GFLOPs if you are matching the DiT paper convention.
- Report total parameters for model size. If your contribution is parameter-efficient fine-tuning, also report trainable parameters.
- Set `benchmark_batch_size` to the real per-GPU batch size before reporting timing or VRAM.
- Set `steps_per_epoch` to your actual number of optimizer steps in one epoch. If you do not know it, set `dataset_size` and `global_batch_size`.
- `benchmark_steps_per_epoch` controls how many live steps are measured. Use the full epoch for the most literal number, or a long representative window for a faster estimate.
- GPU usage is sampled live during the benchmark with `nvidia-smi` when available; PyTorch peak memory is also reported.
- This benchmark measures the model training step on latent tensors. The original training script also includes VAE encoding and data loading, so benchmark those separately if your paper needs full pipeline wall-clock time.
